# CUDA Optimization Course - Module 3: Model Compilation

Use torch.compile() to optimize model execution.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import warnings
warnings.filterwarnings('ignore')

# Device setup
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Device: {device}")
print(f"torch.compile available: {hasattr(torch, 'compile')}")

## Transformer Model

class TransformerModel(nn.Module):
    def __init__(self, vocab_size=10000, d_model=512, nhead=8, num_layers=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, 1024, d_model))
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=2048,
                batch_first=True,
                dropout=0.1
            ),
            num_layers=num_layers
        )
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        seq_len = x.shape[1]
        x = self.embedding(x)
        x = x + self.pos_encoding[:, :seq_len, :]
        x = self.transformer(x)
        x = self.fc(x)
        return x

model = TransformerModel().to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Baseline Performance (Eager Mode)

batch_size = 16
seq_length = 256
num_iterations = 100

def create_batch():
    return torch.randint(0, 10000, (batch_size, seq_length)).to(device)

# Warmup
for _ in range(5):
    with torch.no_grad():
        _ = model(create_batch())

# Measure eager mode
torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

with torch.no_grad():
    for _ in range(num_iterations):
        _ = model(create_batch())

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
eager_time = time.perf_counter() - start

print(f"Eager Mode (Baseline):")
print(f"Time: {eager_time:.2f}s")
print(f"Throughput: {num_iterations * batch_size / eager_time:.0f} samples/sec")
print(f"Time per sample: {eager_time / (num_iterations * batch_size) * 1000:.2f} ms")

## Compiled Model - Reduce Overhead Mode

# Create compiled model
model_compiled = TransformerModel().to(device)

try:
    model_compiled = torch.compile(model_compiled, mode='reduce-overhead')
    print("Model compiled with mode='reduce-overhead'")
except Exception as e:
    print(f"Compilation not supported on {device}: {e}")
    print("Continuing with eager model...")
    model_compiled = model_compiled

# Warmup compiled model
for _ in range(5):
    with torch.no_grad():
        _ = model_compiled(create_batch())

# Measure compiled mode
torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

with torch.no_grad():
    for _ in range(num_iterations):
        _ = model_compiled(create_batch())

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
compiled_time = time.perf_counter() - start

print(f"\nCompiled Mode (reduce-overhead):")
print(f"Time: {compiled_time:.2f}s")
print(f"Throughput: {num_iterations * batch_size / compiled_time:.0f} samples/sec")
print(f"Time per sample: {compiled_time / (num_iterations * batch_size) * 1000:.2f} ms")
print(f"\nSpeedup: {eager_time / compiled_time:.2f}x")

## Compilation Modes

PyTorch provides different compilation modes:

1. **default**: Balanced optimization
2. **reduce-overhead**: Minimizes Python overhead, good for inference
3. **max-autotune**: Maximum optimization, slower compilation

### Mode Comparison

modes = ['reduce-overhead', 'max-autotune']
results = {'eager': eager_time}

for mode in modes:
    try:
        model_test = TransformerModel().to(device)
        print(f"\nCompiling with mode='{mode}'...")
        model_test = torch.compile(model_test, mode=mode)
        
        # Warmup
        for _ in range(5):
            with torch.no_grad():
                _ = model_test(create_batch())
        
        # Measure
        torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
        start = time.perf_counter()
        
        with torch.no_grad():
            for _ in range(num_iterations):
                _ = model_test(create_batch())
        
        torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
        mode_time = time.perf_counter() - start
        results[mode] = mode_time
        
        print(f"Time: {mode_time:.2f}s, Speedup: {eager_time / mode_time:.2f}x")
    except Exception as e:
        print(f"Mode '{mode}' not available: {e}")

print(f"\n=== Summary ===")
for mode, t in sorted(results.items(), key=lambda x: x[1]):
    print(f"{mode:15s}: {t:.2f}s ({eager_time/t:.2f}x speedup)")

## Compile + AMP Combination

Combining compilation with mixed precision for maximum speedup.

from torch.amp import autocast

model_amp = TransformerModel().to(device)

try:
    model_amp = torch.compile(model_amp, mode='reduce-overhead')
except:
    pass

# Warmup
for _ in range(5):
    with torch.no_grad():
        with autocast(device_type=device, dtype=torch.float16):
            _ = model_amp(create_batch())

# Measure compiled + AMP
torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

with torch.no_grad():
    with autocast(device_type=device, dtype=torch.float16):
        for _ in range(num_iterations):
            _ = model_amp(create_batch())

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
amp_compiled_time = time.perf_counter() - start

print(f"Compiled + AMP:")
print(f"Time: {amp_compiled_time:.2f}s")
print(f"Speedup vs Eager: {eager_time / amp_compiled_time:.2f}x")
print(f"Speedup vs Compiled: {compiled_time / amp_compiled_time:.2f}x")

## Key Takeaways

1. **torch.compile()**: Optimizes the computation graph for your hardware
2. **Reduce Overhead**: Best for inference, minimizes Python overhead
3. **Max Autotune**: Slower compilation but maximum runtime performance
4. **Hardware Specific**: Compilation adapts to your device (CUDA/MPS/CPU)
5. **Best Results**: Combine with AMP for maximum speedup
6. **Warmup Required**: First runs trigger compilation, subsequent runs are fast